# Consultas · Open Food Facts

Preguntas 1 a 11 sobre el Parquet limpio (`/data/clean/off_limpio`), ejecutadas en `hadoop-client-consulta` con Spark sobre YARN.
Una sola sesión de Spark para todo el notebook: ejecutar primero la celda de preparación y después la sección de cada pregunta, en orden.

| Sección | Pregunta |
|---|---|
| [P1](#pregunta-1--vegano--vegetariano-por-categoría) | Vegano / vegetariano por categoría |
| [P2](#pregunta-2--productos-altos-en-azúcar-según-el-semáforo-nutricional) | Productos altos en azúcar |
| [P3](#pregunta-3--distribución-del-nutri-score-por-país) | Nutri-Score por país |
| [P4](#pregunta-4--aditivos-frente-a-nivel-de-procesamiento-nova) | Aditivos frente a NOVA |
| [P5](#pregunta-5--marcas-con-mejor-y-peor-perfil-nutricional-dentro-de-cada-grupo) | Marcas con mejor y peor perfil nutricional dentro de cada grupo (Alberto) — pendiente |
| [P6](#pregunta-6--evolución-anual-de-la-cantidad-y-calidad-de-los-datos) | Evolución anual de la cantidad y calidad de los datos (Alberto) — pendiente |
| [P7](#pregunta-7--coherencia-entre-la-energía-declarada-y-la-calculada-por-país) | Coherencia entre la energía declarada y la calculada, por país (Alberto) — pendiente |
| [P8](#pregunta-8--alegaciones-de-proteínas-y-fibra-reglamento-ue-19242006) | Alegaciones de proteínas y fibra (Reglamento UE 1924/2006) (Jorge) — pendiente |
| [P9](#pregunta-9--aditivos-más-frecuentes-y-su-presencia-por-país) | Aditivos más frecuentes y su presencia por país (Jorge) — pendiente |
| [P10](#pregunta-10--mejores-categorías-como-fuente-de-hierro-calcio-y-vitamina-c) | Mejores categorías como fuente de hierro, calcio y vitamina C (Jorge) — pendiente |
| [P11](#pregunta-11--reparto-de-la-energía-entre-grasas-carbohidratos-y-proteínas) | Reparto de la energía entre grasas, carbohidratos y proteínas (Jorge) — pendiente |

Las tablas se guardan en `/results/pNN_…` (HDFS) y las gráficas en `figuras/`.

**Para quien implemente una pregunta:** cada sección pendiente trae el enunciado, las columnas y las operaciones sugeridas. Rellenar las celdas `TODO` en orden (consulta, tabla, gráfica) y escribir la conclusión solo después de interpretar la gráfica.

## Preparación

In [ ]:
import os

# pyspark (de pip) toma los jars y spark-defaults.conf (yarn, modo client) de /opt/spark.
# Se fijan aquí las variables del cliente de consulta: el kernel no siempre hereda ~/.bashrc.
os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["HADOOP_USER_NAME"] = "consulta"
os.environ["HADOOP_CONF_DIR"] = os.path.expanduser("~/hadoop_conf")
os.environ["YARN_CONF_DIR"] = os.path.expanduser("~/hadoop_conf")

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F

ENTRADA = "hdfs://hadoop-master:9000/data/clean/off_limpio"
SALIDA = "hdfs://hadoop-master:9000/results"
os.makedirs("figuras", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print("Usuario de HDFS:", os.environ.get("HADOOP_USER_NAME"), "(debe ser: consulta)")
spark = SparkSession.builder.appName("consulta_preguntas").getOrCreate()
df = spark.read.parquet(ENTRADA)
print("Modo de Spark:", spark.sparkContext.master, "(debe ser: yarn)")
print("Filas en el Parquet limpio:", df.count())

def cuenta(condicion):
    """Número de filas que cumplen la condición (las nulas no cuentan)."""
    return F.sum(F.when(condicion, 1).otherwise(0))

## Pregunta 1 · Vegano / vegetariano por categoría

**Enunciado:** qué porcentaje de productos son veganos o vegetarianos y cómo varía según la categoría de alimento.

- **Columnas:** `es_vegano`, `es_vegetariano`, `categoria_pnns`.
- **Criterio:** el % se calcula sobre los productos con el estado **conocido** (`true` o `false`); los de estado nulo se cuentan aparte.
- **Ejecución:** en `hadoop-client-consulta` (Spark sobre YARN).

### Consulta

In [ ]:
resultado = (df.filter(F.col("categoria_pnns").isNotNull())
               .groupBy("categoria_pnns")
               .agg(F.count("*").alias("n_total"),
                    cuenta(F.col("es_vegano") == True).alias("vegano_si"),
                    cuenta(F.col("es_vegano") == False).alias("vegano_no"),
                    cuenta(F.col("es_vegetariano") == True).alias("vegetariano_si"),
                    cuenta(F.col("es_vegetariano") == False).alias("vegetariano_no"))
               .orderBy(F.col("n_total").desc()))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results)
resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p01_vegano_vegetariano")

### Tabla en pandas

In [ ]:
p1 = resultado.toPandas()                      # unas decenas de filas
p1["vegano_conocido"] = p1.vegano_si + p1.vegano_no
p1["vegetariano_conocido"] = p1.vegetariano_si + p1.vegetariano_no
p1["pct_vegano"] = 100 * p1.vegano_si / p1.vegano_conocido
p1["pct_vegetariano"] = 100 * p1.vegetariano_si / p1.vegetariano_conocido

print("Productos con estado vegano conocido:", int(p1.vegano_conocido.sum()), "de", int(p1.n_total.sum()))
print("Productos con estado vegetariano conocido:", int(p1.vegetariano_conocido.sum()), "de", int(p1.n_total.sum()))
p1.round(1)

### Gráfica

In [ ]:
MIN_CATEGORIA = 500    # categorías con menos productos con estado conocido se descartan
g = (p1[p1.vegano_conocido >= MIN_CATEGORIA].sort_values("pct_vegano").tail(15)
       .set_index("categoria_pnns")[["pct_vegano", "pct_vegetariano"]])

ax = g.plot.barh(figsize=(9, 6), color=["#2e8b57", "#9acd32"])
ax.set_xlabel("% de productos (sobre los que tienen el estado conocido)")
ax.set_ylabel("")
ax.set_title("P1 · Productos veganos y vegetarianos por categoría (top 15 por % vegano)")
ax.legend(["Vegano", "Vegetariano"], frameon=False)
plt.tight_layout()
plt.savefig("figuras/p01_vegano_vegetariano.png")
plt.show()

### Conclusión

Los productos veganos se concentran en las categorías de origen vegetal. Aguas, zumos, fruta, legumbres, frutos secos y verduras superan el 90 % de productos veganos entre los que declaran su estado.

En las categorías donde puede haber leche, huevo o grasa animal, baja bastante. Los cereales están en un 66 % vegano, las grasas en un 63 % y las patatas en un 57 %, aunque casi todos esos productos son vegetarianos (≈ 90 % o más).

La diferencia entre las dos barras son los productos vegetarianos que no son veganos, es decir, los que llevan lácteos, huevo o miel.

Limitación: estos porcentajes se calculan solo sobre los productos que declaran su estado, que son ≈ 38 % de los que tienen categoría. Los demás no se sabe si son veganos o no.

In [ ]:
# por curisidad hemos filtrado los productos que son frutas pero no son veganos
# esto nos permite ver qué aditivos contienen estos productos
(df.filter((F.col("categoria_pnns") == "Fruits") & (F.col("es_vegano") == False))
   .select("product_name", "marca_principal", "additives_tags")
   .show(30, truncate=False))

# Ni siquiera la fruta llega al 100 %: la categoría incluye productos elaborados 
# (con nata o yogur) y fruta con aditivos de origen animal, como el carmín o la cera de goma laca.

## Pregunta 2 · Productos altos en azúcar según el semáforo nutricional

**Enunciado:** qué porcentaje de productos superan 22,5 g de azúcar por 100 g (Reglamento UE 1169/2011) y qué categorías concentran más casos.

- **Columnas:** `sugars_100g`, `categoria_pnns`.
- **Criterio:** se excluyen las filas con `nutrientes_incoherentes`. El umbral de 22,5 g/100 g se aplica también a líquidos (cuyo umbral es 27 g/100 ml): es una simplificación que debe constar en la respuesta.

### Consulta

In [ ]:
UMBRAL_AZUCAR = 22.5   # g/100 g

resultado = (df.filter(F.col("sugars_100g").isNotNull()
                       & F.col("categoria_pnns").isNotNull()
                       & ~F.col("nutrientes_incoherentes"))
               .groupBy("categoria_pnns")
               .agg(F.count("*").alias("n_productos"),
                    cuenta(F.col("sugars_100g") > UMBRAL_AZUCAR).alias("n_alto_azucar"),
                    F.round(F.avg("sugars_100g"), 2).alias("azucar_media_100g"))
               .orderBy(F.col("n_productos").desc()))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results)
resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p02_alto_azucar")

### Tabla en pandas

In [ ]:
p2 = resultado.toPandas()
p2["pct_alto"] = 100 * p2.n_alto_azucar / p2.n_productos
total = 100 * p2.n_alto_azucar.sum() / p2.n_productos.sum()
print(f"Altos en azúcar en total: {total:.1f} % de {int(p2.n_productos.sum())} productos")
p2.round(1)

### Gráfica

In [ ]:
MIN_CATEGORIA = 500
g = p2[p2.n_productos >= MIN_CATEGORIA].sort_values("pct_alto").tail(15).set_index("categoria_pnns").pct_alto

ax = g.plot.barh(figsize=(9, 6), color="#d2691e")
ax.axvline(total, color="gray", linestyle="--", label=f"Total: {total:.1f} %")
ax.set_xlabel("% de productos con más de 22,5 g de azúcar por 100 g")
ax.set_ylabel("")
ax.set_title("P2 · Categorías con más productos altos en azúcar (top 15)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("figuras/p02_alto_azucar.png")
plt.show()

### Conclusión

Los productos más altos en azúcar son los dulces y el chocolate, más del 80% por encima del umbral de 22,5 g por 100 g, seguidos de la fruta seca (≈ 83 %), las galletas y pasteles (≈ 71 %) y los helados (≈ 58 %). En total, ≈ 21 % de los productos supera ese umbral.

Los alimentos infantiles (Baby foods) tienen pocos productos altos en azúcar (≈ 8 %), pero las leches infantiles suben a ≈ 28 %. Esto se debe a la lactosa de la leche en polvo, que es azúcar natural, y no a azúcar añadido.

Limitaciones: el criterio usa el azúcar total, así que la fruta seca sale alta por su azúcar natural. Además, el umbral por 100 g no está pensado para bebidas, de modo que las bebidas azucaradas aparecen con porcentajes bajos.

## Pregunta 3 · Distribución del Nutri-Score por país

**Enunciado:** cómo se reparte la calificación Nutri-Score (A-E) entre los países donde se vende el producto.

- **Columnas:** `nutriscore_grade`, `paises`.
- **Criterio:** un producto cuenta en cada país donde se vende (`explode`). Solo se conservan los países con al menos 1.000 productos.

### Consulta

In [ ]:
MIN_PRODUCTOS_PAIS = 1000

resultado = (df.filter(F.col("nutriscore_grade").isNotNull() & F.col("paises").isNotNull())
               .select(F.explode("paises").alias("pais"), "nutriscore_grade")
               .groupBy("pais").pivot("nutriscore_grade", ["a", "b", "c", "d", "e"]).count()
               .fillna(0)
               .withColumn("total", F.col("a") + F.col("b") + F.col("c") + F.col("d") + F.col("e"))
               .filter(F.col("total") >= MIN_PRODUCTOS_PAIS)
               .orderBy(F.col("total").desc()))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results)
resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p03_nutriscore_pais")

### Tabla en pandas

In [ ]:
p3 = resultado.toPandas()
p3["pais"] = p3.pais.str.replace(r"^[a-z]{2}:", "", regex=True).str.replace("-", " ").str.title()
print("Países con al menos", MIN_PRODUCTOS_PAIS, "productos:", len(p3))
p3.head(20)

### Gráfica

In [ ]:
letras = ["a", "b", "c", "d", "e"]
colores = ["#038141", "#85bb2f", "#fecb02", "#ee8100", "#e63e11"]   # paleta oficial del Nutri-Score

top = p3.sort_values("total", ascending=False).head(30).copy()
pct = top[letras].div(top.total, axis=0) * 100
pct.index = top.pais
pct = pct.sort_values("a")          # los países con más A quedan arriba

ax = pct.plot.barh(stacked=True, figsize=(9, 6), color=colores, width=0.8)
ax.set_xlim(0, 100)
ax.set_xlabel("% de productos")
ax.set_ylabel("")
ax.set_title("P3 · Nutri-Score por país (30 países con más productos)")
ax.legend([l.upper() for l in letras], ncol=5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.savefig("figuras/p03_nutriscore_pais.png")
plt.show()

### Conclusión

Los productos con Nutri-Score A o B son una minoría en prácticamente todos los países: cerca de 1 de cada 3. Y los de calificación D o E son ≈ la mitad. El reparto es parecido de un país a otro, lo que sugiere que la calidad nutricional de los alimentos envasados depende más del tipo de producto que del país donde se vende.

Hay excepciones. Brasil tiene más productos buenos (≈ 40 % entre A y B), y India, Marruecos y Tailandia tienen más malos (≈ 57-64 % entre D y E).

## Pregunta 4 · Aditivos frente a nivel de procesamiento (NOVA)

**Enunciado:** a mayor número de aditivos (`additives_n`), ¿es mayor la probabilidad de que el producto esté en el grupo NOVA 4 (ultraprocesado)?

- **Columnas:** `additives_n`, `nova_group`.
- **Criterio:** se agrupan los valores altos (10 significa "10 o más"). NOVA 4 se define en parte por los aditivos, así que la relación es en parte circular y debe señalarse.

### Consulta

In [ ]:
TOPE_ADITIVOS = 10   # 10 significa "10 o más"

resultado = (df.filter(F.col("additives_n").isNotNull() & F.col("nova_group").isNotNull())
               .withColumn("aditivos", F.least(F.col("additives_n"), F.lit(TOPE_ADITIVOS)))
               .groupBy("aditivos").pivot("nova_group", [1, 2, 3, 4]).count()
               .fillna(0)
               .orderBy("aditivos"))
for n in (1, 2, 3, 4):
    resultado = resultado.withColumnRenamed(str(n), f"nova_{n}")

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results)
resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p04_aditivos_nova")

### Tabla en pandas

In [ ]:
p4 = resultado.toPandas()
p4["total"] = p4[["nova_1", "nova_2", "nova_3", "nova_4"]].sum(axis=1)
p4["pct_nova4"] = 100 * p4.nova_4 / p4.total
p4.round(1)

### Gráfica

In [ ]:
etiquetas = [str(a) if a < TOPE_ADITIVOS else f"{TOPE_ADITIVOS} o más" for a in p4.aditivos]
fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar(etiquetas, p4.pct_nova4, color="#8b3a3a")
ax.bar_label(barras, fmt="%.0f %%", fontsize=8)
ax.set_ylim(0, 105)
ax.set_xlabel("Número de aditivos del producto")
ax.set_ylabel("% de productos en NOVA 4 (ultraprocesado)")
ax.set_title("P4 · Más aditivos, más ultraprocesados")
plt.tight_layout()
plt.savefig("figuras/p04_aditivos_nova.png")
plt.show()

### Conclusión

Los productos con más aditivos tienen mucha más probabilidad de ser ultraprocesados: pasa del 25 % entre los que no tienen ninguno al 78 % con uno, al 97 % con tres, y a prácticamente el 100 % con cinco o más.

Esta relación es en parte circular, porque la clasificación NOVA 4 incluye la presencia de aditivos como criterio. Por eso lo más informativo es que 1 de cada 4 productos sin aditivos también es ultraprocesado, lo que indica que los aditivos no son la única señal.

Limitación: solo se analizan los productos que tienen informado tanto el número de aditivos como el grupo NOVA (≈ 23 % del total).

## Pregunta 5 · Marcas con mejor y peor perfil nutricional dentro de cada grupo

> **Pendiente de implementar** · Responsable: Alberto

**Enunciado:** dentro de cada grupo PNNS, ¿qué 5 marcas con al menos N productos tienen la media más baja y la más alta de azúcar, grasa saturada y sal?

- **Columnas:** `marcas`, `grupo_pnns`, `sugars_100g`, `saturated_fat_100g`, `salt_100g`.
- **Operaciones de Spark sugeridas:** `explode` de `marcas`, `groupBy`/`agg` (`avg`, `count`), filtro por mínimo de productos, ventana `row_number()` con `Window.partitionBy(...).orderBy(...)`.
- **Ojo con los datos:** dificultad 3. Marcas blancas (`hacendado`, `marque-repere`, `u`) cuentan como marca. Excluir filas con `nutrientes_incoherentes`. Decidir y anotar el mínimo de productos por marca.
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p05_marcas_perfil")

### Tabla en pandas

In [ ]:
# TODO: p5 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p05_marcas_perfil.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Pregunta 6 · Evolución anual de la cantidad y calidad de los datos

> **Pendiente de implementar** · Responsable: Alberto

**Enunciado:** por año de alta, ¿cuántos productos se añadieron, qué porcentaje tiene energía y macronutrientes informados y cómo crece cada año respecto al anterior?

- **Columnas:** `anio_alta`, `energy_kcal_100g`, `fat_100g`, `carbohydrates_100g`, `proteins_100g`.
- **Operaciones de Spark sugeridas:** `groupBy`/`agg` con `avg(when(col.isNotNull(), 1).otherwise(0))`, ventana con `lag` para la variación interanual.
- **Ojo con los datos:** dificultad 2. La energía y los macronutrientes tienen un 28-29 % de nulos, que es justo la métrica medida. 2026 es un año incompleto y debe indicarse.
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p06_evolucion_anual")

### Tabla en pandas

In [ ]:
# TODO: p6 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p06_evolucion_anual.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Pregunta 7 · Coherencia entre la energía declarada y la calculada, por país

> **Pendiente de implementar** · Responsable: Alberto

**Enunciado:** ¿qué porcentaje de productos tiene una energía declarada que difiere más de un 20 % de la calculada con `4·carbohidratos + 4·proteínas + 9·grasa`, y cómo varía según el país?

- **Columnas:** `energy_kcal_100g`, `carbohydrates_100g`, `proteins_100g`, `fat_100g`, `paises`.
- **Operaciones de Spark sugeridas:** `withColumn` con expresión aritmética, `when`/`otherwise`, `explode` de `paises`, `groupBy`/`agg`, filtro por mínimo de productos por país.
- **Ojo con los datos:** dificultad 2. Medir cuántas filas tienen las cuatro columnas a la vez (cada una ronda el 28 % de nulos). Un producto cuenta en cada país donde se vende.
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p07_energia_coherencia")

### Tabla en pandas

In [ ]:
# TODO: p7 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p07_energia_coherencia.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Pregunta 8 · Alegaciones de proteínas y fibra (Reglamento UE 1924/2006)

> **Pendiente de implementar** · Responsable: Jorge

**Enunciado:** ¿qué porcentaje de productos cumple los umbrales legales para declararse «fuente de proteínas» o «alto contenido de proteínas», y «fuente de fibra» o «alto contenido de fibra», y en qué grupos se concentran?

- **Columnas:** `proteins_100g`, `energy_kcal_100g`, `fiber_100g`, `grupo_pnns`.
- **Operaciones de Spark sugeridas:** `withColumn` para el % de energía de las proteínas, `when`/`otherwise` para asignar cada alegación, `groupBy`/`agg` por grupo.
- **Ojo con los datos:** dificultad 2. Umbrales: fuente de proteínas ≥ 12 % de la energía (`proteins_100g · 4 / energy_kcal_100g`), alto ≥ 20 %; fuente de fibra ≥ 3 g/100 g, alto ≥ 6 g/100 g. `fiber_100g` tiene un 61 % de nulos: la parte de fibra se calcula solo sobre los que la informan.
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p08_alegaciones")

### Tabla en pandas

In [ ]:
# TODO: p8 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p08_alegaciones.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Pregunta 9 · Aditivos más frecuentes y su presencia por país

> **Pendiente de implementar** · Responsable: Jorge

**Enunciado:** ¿cuáles son los 10 aditivos (E-números) más usados y en qué países aparecen proporcionalmente más?

- **Columnas:** `additives_tags`, `paises`.
- **Operaciones de Spark sugeridas:** doble `explode` (aditivos × países), `groupBy`/`count`, normalización por el total de productos con aditivos informados de cada país, filtro por mínimo de productos, ventana `row_number()` con `Window.partitionBy("pais")` para el top N.
- **Ojo con los datos:** dificultad 3. El doble `explode` multiplica las filas: conviene `repartition` y vigilar la memoria de los executors. Distinguir «sin aditivos» de «no informado» (≈ 72 % de nulos).
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p09_aditivos_pais")

### Tabla en pandas

In [ ]:
# TODO: p9 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p09_aditivos_pais.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Pregunta 10 · Mejores categorías como fuente de hierro, calcio y vitamina C

> **Pendiente de implementar** · Responsable: Jorge

**Enunciado:** ¿qué categorías aportan más hierro, calcio y vitamina C por 100 g y por 100 kcal, y qué proporción de sus productos informa esos nutrientes?

- **Columnas:** `iron_100g`, `calcium_100g`, `vitamin_c_100g`, `energy_kcal_100g`, `categories_tags`.
- **Operaciones de Spark sugeridas:** `explode` de categorías, `withColumn` para el valor por 100 kcal, `groupBy`/`agg` (`avg`, `percentile_approx`, `count`, % de no nulos), filtro por mínimo de productos con dato, `rank` con ventana por nutriente.
- **Ojo con los datos:** dificultad 2. Riesgo de datos alto: hierro 84 %, calcio 84,2 % y vitamina C 88,8 % de nulos. Interpretar solo sobre los productos que informan el nutriente; el % de cobertura forma parte de la respuesta. `categories_tags` es una lista separada por comas: hacer `split` antes de usarla.
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p10_micronutrientes")

### Tabla en pandas

In [ ]:
# TODO: p10 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p10_micronutrientes.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Pregunta 11 · Reparto de la energía entre grasas, carbohidratos y proteínas

> **Pendiente de implementar** · Responsable: Jorge

**Enunciado:** ¿qué porcentaje de los productos de cada categoría reparte su energía dentro de los rangos de referencia (grasa 20-35 % y carbohidratos 45-60 % de la energía, según EFSA/OMS) y cuáles se desvían más?

- **Columnas:** `fat_100g`, `carbohydrates_100g`, `proteins_100g`, `energy_kcal_100g`, `categories_tags`.
- **Operaciones de Spark sugeridas:** `withColumn` para el % de energía de cada macronutriente (`9·grasa`, `4·carbohidratos`, `4·proteínas`, sobre la suma), filtro de filas incoherentes (`nutrientes_incoherentes`), `when`/`otherwise` para «dentro/fuera de rango», `explode` de categorías, `groupBy`/`agg`.
- **Ojo con los datos:** dificultad 2. Medir las filas con los cuatro nutrientes a la vez. Confirmar la fuente exacta de los rangos antes de publicar.
- **Detalle completo:** [docs/PREGUNTAS.md](../../docs/PREGUNTAS.md).

### Consulta

In [ ]:
# TODO: escribir la consulta. Debe dejar un DataFrame de Spark llamado `resultado` (una tabla pequeña).
# resultado = (df.filter(...)
#                .groupBy(...)
#                .agg(...))

# Guardar la tabla en HDFS (solo el usuario consulta puede escribir en /results):
# resultado.coalesce(1).write.mode("overwrite").parquet(f"{SALIDA}/p11_macronutrientes")

### Tabla en pandas

In [ ]:
# TODO: p11 = resultado.toPandas() y calcular las columnas derivadas (porcentajes, etc.)

### Gráfica

In [ ]:
# TODO: gráfica con matplotlib (título, ejes con unidades) y guardarla:
# plt.tight_layout()
# plt.savefig("figuras/p11_macronutrientes.png")
# plt.show()

### Conclusión

_(Escribir cuando ya se haya interpretado la gráfica: de 2 a 4 frases sencillas, con las cifras que se ven en la gráfica y las limitaciones de los datos.)_

## Cerrar

In [ ]:
# Liberar los executors de YARN: si no, este notebook bloquea a los demás trabajos
spark.stop()